### SFOS API - IPv4 Address Groups Management

**Purpose:** Create, read, update, and delete IPv4 address groups

**Topics:**
- Basic CRUD operations using domain StrEnum objects
- Group membership management
- Pagination and filtering
- Error handling
- Bulk operations

**Requirements:**
- Environment variables configured (.env file)
- SFOS firewall accessible
- API key with appropriate permissions

---

#### Setup: Import Libraries and Configure Client

In [21]:
from common import print_data, setup_client

from sfos_api import Network

# Initialize the SFOS API Client
client = setup_client()
print("✅ SFOS API client initialized")

✅ SFOS API client initialized


#### Create IPv4 Address Groups

Create IPv4 address group objects using the `Network.ADDRESS_GROUPS_IPV4` endpoint object.

Address groups allow you to combine multiple IPv4 addresses or networks for use in firewall rules.

API Link: [Create an IPv4 group | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/create-ipv-4-address-group/)

In [22]:
# Create an IPv4 address group
endpoint = Network.ADDRESS_GROUPS_IPV4
group_name = "test-ipv4-group-internal"

payload = {
    "name": group_name,
    "description": "Internal network addresses",
    "ipv4Addresses": [
        {"name": "test-ipv4address-one"},
        {"name": "test-ipv4address-two"},
    ],
}

try:
    response = client.create(endpoint, payload)
    print(f"✅ Created IPv4 address group '{group_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating IPv4 address group: {e}")
    print_data(str(e))

✅ Created IPv4 address group 'test-ipv4-group-internal'
{
  "id": "c84dfad5-f91e-56cf-b37e-992467cca49e",
  "name": "test-ipv4-group-internal",
  "description": "Internal network addresses",
  "ipv4Addresses": [
    {
      "name": "test-ipv4address-one"
    },
    {
      "name": "test-ipv4address-two"
    }
  ],
  "createdAt": "2026-10-05T08:31:52.197Z",
  "updatedAt": "2026-10-05T08:31:52.197Z"
}


In [23]:
# Create another IPv4 address group
endpoint = Network.ADDRESS_GROUPS_IPV4
group_name = "test-ipv4-group-external"

payload = {
    "name": group_name,
    "description": "External network addresses",
    "ipv4Addresses": [
        {"name": "test-ipv4network"},
    ],
}

try:
    response = client.create(endpoint, payload)
    print(f"✅ Created IPv4 address group '{group_name}'")
    print_data(response)
except Exception as e:
    print(f"Error creating IPv4 address group: {e}")
    print_data(str(e))

✅ Created IPv4 address group 'test-ipv4-group-external'
{
  "id": "9a7feec2-3fbb-5a65-980d-a1ed0a585842",
  "name": "test-ipv4-group-external",
  "description": "External network addresses",
  "ipv4Addresses": [
    {
      "name": "test-ipv4network"
    }
  ],
  "createdAt": "2026-10-05T08:31:53.472Z",
  "updatedAt": "2026-10-05T08:31:53.472Z"
}


#### Read IPv4 Address Groups

Retrieve IPv4 address group information by ID/name or list all address groups.

API Links: [Get IPv4 address groups | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/get-all-ipv-4-address-groups/) | [Get an IPv4 group | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/get-ipv-4-address-group-by-id-or-name/)

In [24]:
# Get a specific IPv4 address group by name
endpoint = Network.ADDRESS_GROUPS_IPV4
group_name = "test-ipv4-group-internal"

try:
    response = client.read(endpoint, id_or_name=group_name)
    print(f"✅ Retrieved IPv4 address group '{group_name}'")
    print_data(response, keys=["name", "description", "ipv4Addresses"])
except Exception as e:
    print(f"Error reading IPv4 address group: {e}")
    print_data(str(e))

✅ Retrieved IPv4 address group 'test-ipv4-group-internal'
{
  "name": "test-ipv4-group-internal",
  "description": "Internal network addresses",
  "ipv4Addresses": [
    {
      "name": "test-ipv4address-one"
    },
    {
      "name": "test-ipv4address-two"
    }
  ]
}


In [25]:
# Get all IPv4 address groups
endpoint = Network.ADDRESS_GROUPS_IPV4

try:
    response = client.read(endpoint)

    if "items" in response and response["items"]:
        print(f"✅ Retrieved {len(response['items'])} IPv4 address groups")
        # Show first 5 groups
        print_data(response["items"][:5], keys=["name", "description", "ipv4Addresses"])
        if len(response["items"]) > 5:
            print(f"... (showing first 5 of {len(response['items'])} groups)")
    else:
        print("No IPv4 address groups found")
except Exception as e:
    print(f"Error reading IPv4 address groups: {e}")
    print_data(str(e))

✅ Retrieved 3 IPv4 address groups
1/3 : {"description": "Unreserved public IP addresses","ipv4Addresses": [{"name": "Internet IPv4 (1-9)"},{"name": "Internet IPv4 (11-126)"},{"name": "Internet IPv4 (128-169)"},{"name": "Internet IPv4 (169-172)"},{"name": "Internet IPv4 (172-191)"},{"name": "Internet IPv4 (191-191.1)"},{"name": "Internet IPv4 (192-192.0)"},{"name": "Internet IPv4 (192-192.88)"},{"name": "Internet IPv4 (192.169-198.17)"},{"name": "Internet IPv4 (192.88-192.167)"},{"name": "Internet IPv4 (198.20-198.51)"},{"name": "Internet IPv4 (198.51-203.0)"},{"name": "Internet IPv4 (203.0-223)"}],"name": "Internet IPv4 group"}
2/3 : {"description": "External network addresses","ipv4Addresses": [{"name": "test-ipv4network"}],"name": "test-ipv4-group-external"}
3/3 : {"description": "Internal network addresses","ipv4Addresses": [{"name": "test-ipv4address-one"},{"name": "test-ipv4address-two"}],"name": "test-ipv4-group-internal"}


In [26]:
# Get IPv4 address groups with filtering
endpoint = Network.ADDRESS_GROUPS_IPV4
params = {
    "nameContains": "test-ipv4-group-",
}

try:
    response = client.read(endpoint, params=params)

    if "items" in response:
        print(f"✅ Retrieved {len(response['items'])} IPv4 address groups containing 'test-ipv4-group-'")
        print_data(response["items"], keys=["name", "description"])
    else:
        print("No IPv4 address groups found matching filter")
except Exception as e:
    print(f"Error reading IPv4 address groups: {e}")
    print_data(str(e))

✅ Retrieved 3 IPv4 address groups containing 'test-ipv4-group-'
1/3 : {"description": "Unreserved public IP addresses","name": "Internet IPv4 group"}
2/3 : {"description": "External network addresses","name": "test-ipv4-group-external"}
3/3 : {"description": "Internal network addresses","name": "test-ipv4-group-internal"}


#### Update IPv4 Address Group

Modify an existing IPv4 address group's properties.

API Link: [Update an IPv4 group | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/update-ipv-4-address-group-by-id-or-name/)

In [27]:
# Update an IPv4 address group
endpoint = Network.ADDRESS_GROUPS_IPV4
group_name = "test-ipv4-group-internal"

payload = {
    "description": "Internal network addresses - including all internal subnets",
    "ipv4Addresses": [
        {"name": "test-ipv4address-one"},
        {"name": "test-ipv4address-two"},
        {"name": "test-ipv4network"},
    ],
}

try:
    response = client.update(endpoint, id_or_name=group_name, data=payload)
    print(f"✅ Updated IPv4 address group '{group_name}'")
    print_data(response)
except Exception as e:
    print(f"Error updating IPv4 address group: {e}")
    print_data(str(e))

✅ Updated IPv4 address group 'test-ipv4-group-internal'
{
  "id": "c84dfad5-f91e-56cf-b37e-992467cca49e",
  "name": "test-ipv4-group-internal",
  "description": "Internal network addresses - including all internal subnets",
  "ipv4Addresses": [
    {
      "name": "test-ipv4address-one"
    },
    {
      "name": "test-ipv4address-two"
    },
    {
      "name": "test-ipv4network"
    }
  ],
  "createdAt": "2026-10-05T08:31:52.197Z",
  "updatedAt": "2026-10-05T08:31:55.392Z"
}


#### Delete IPv4 Address Groups

Remove IPv4 address group objects (single or bulk delete).

API Links: [Delete an IPv4 group | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/delete-ipv-4-address-group-by-id-or-name/) | [Bulk delete IPv4 groups | Sophos Firewall API](https://docs.sophos.com/nsg/sophos-firewall/rest-api/api/firewall-config-openapi/bulk-delete-ipv-4-address-groups/)

In [ ]:
# Delete a single IPv4 address group
endpoint = Network.ADDRESS_GROUPS_IPV4
group_name = "test-ipv4-group-internal"

try:
    response = client.delete(endpoint, id_or_name=group_name)
    print(f"✅ Deleted IPv4 address group '{group_name}'")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error deleting IPv4 address group: {e}")
    print_data(str(e))

In [ ]:
# Bulk delete IPv4 address groups
endpoint = Network.ADDRESS_GROUPS_IPV4

items_to_delete = [
    {"name": "test-ipv4-group-internal"},
    {"name": "test-ipv4-group-external"},
]

try:
    response = client.delete(endpoint, items=items_to_delete)
    print(f"✅ Bulk deleted {len(items_to_delete)} IPv4 address groups")
    if response:
        print_data(response)
except Exception as e:
    print(f"Error bulk deleting IPv4 address groups: {e}")
    print_data(str(e))

In [ ]:
# Close the client session when done
client.close()
print("✅ Client session closed")